In [1]:
"""
Live Color-Tracking Object Detector
-----------------------------------
Tracks a colored object (a ball, sticky note, marker cap...) in your webcam
feed and draws a circle, a center dot, and a motion trail on it.

Install:   pip install opencv-python numpy
Run:       python color_tracker.py

Controls (click the video window first so it has focus):
  1-5 : choose color  (1 blue, 2 green, 3 red, 4 yellow, 5 orange)
  t   : toggle HSV tuning sliders (fine-tune the range live)
  m   : show / hide the mask window (what the computer "sees")
  c   : clear the trail
  q   : quit

How it works (the pipeline each frame):
  1. Capture a frame           -> a grid of BGR pixels
  2. Blur it                   -> removes tiny noise
  3. Convert BGR -> HSV        -> separates color (hue) from brightness
  4. inRange()                 -> mask: white where pixels match our color
  5. erode + dilate            -> cleans specks and fills small holes
  6. findContours()            -> outlines of the white blobs
  7. Pick the largest blob     -> that's our object; draw on it
"""

from collections import deque

import cv2
import numpy as np

# ---------- Settings you can change ----------

# HSV ranges. OpenCV hue runs 0-179 (not 0-360), saturation and value 0-255.
# Each color is a LIST of (lower, upper) ranges, because red wraps around
# the hue wheel and needs two ranges.
COLOR_PRESETS = {
    "blue":   [((100, 120, 70), (130, 255, 255))],
    "green":  [((40, 80, 70),   (85, 255, 255))],
    "red":    [((0, 120, 70),   (10, 255, 255)),
               ((170, 120, 70), (179, 255, 255))],
    "yellow": [((20, 100, 100), (35, 255, 255))],
    "orange": [((10, 120, 100), (20, 255, 255))],
}
KEY_TO_COLOR = {
    ord("1"): "blue", ord("2"): "green", ord("3"): "red",
    ord("4"): "yellow", ord("5"): "orange",
}
# BGR drawing color to match each tracked color
DRAW_COLORS = {
    "blue": (255, 120, 0), "green": (0, 200, 0), "red": (0, 0, 255),
    "yellow": (0, 230, 230), "orange": (0, 140, 255),
}

MIN_AREA = 500        # ignore blobs smaller than this many pixels
TRAIL_LENGTH = 40     # how many past positions the trail remembers
CAMERA_INDEX = 0      # change to 1 if you have more than one camera


# ---------- Core vision functions ----------

def build_mask(hsv_frame, ranges):
    """Return a black/white image: white where the pixel is inside any range."""
    mask = np.zeros(hsv_frame.shape[:2], dtype=np.uint8)
    for lower, upper in ranges:
        mask |= cv2.inRange(hsv_frame, np.array(lower), np.array(upper))

    # Morphology: erode removes tiny white specks, dilate restores the size
    # of the real object and closes small holes.
    mask = cv2.erode(mask, None, iterations=2)
    mask = cv2.dilate(mask, None, iterations=2)
    return mask


def find_largest_object(mask, min_area=MIN_AREA):
    """Return ((x, y), radius, contour) of the biggest blob, or None."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    largest = max(contours, key=cv2.contourArea)
    if cv2.contourArea(largest) < min_area:
        return None

    (x, y), radius = cv2.minEnclosingCircle(largest)
    return (int(x), int(y)), int(radius), largest


def process_frame(frame, ranges):
    """Run the whole pipeline. Returns (center, radius, mask) for one frame."""
    blurred = cv2.GaussianBlur(frame, (11, 11), 0)
    hsv = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)   # the key conversion
    mask = build_mask(hsv, ranges)
    found = find_largest_object(mask)
    if found is None:
        return None, None, mask
    center, radius, _ = found
    return center, radius, mask


def draw_trail(frame, trail, color):
    """Draw lines between recent positions; older segments are thinner."""
    for i in range(1, len(trail)):
        if trail[i - 1] is None or trail[i] is None:
            continue
        thickness = max(1, int(4 * i / len(trail)))
        cv2.line(frame, trail[i - 1], trail[i], color, thickness)


# ---------- Optional tuning sliders ----------

def open_trackbars(ranges):
    """Create sliders initialised from the first range of the current color."""
    (h1, s1, v1), (h2, s2, v2) = ranges[0]
    cv2.namedWindow("Tuning")
    for name, value, maximum in [
        ("H low", h1, 179), ("S low", s1, 255), ("V low", v1, 255),
        ("H high", h2, 179), ("S high", s2, 255), ("V high", v2, 255),
    ]:
        cv2.createTrackbar(name, "Tuning", value, maximum, lambda _: None)


def read_trackbars():
    """Read the sliders and return them as a single custom range list."""
    get = lambda n: cv2.getTrackbarPos(n, "Tuning")
    lower = (get("H low"), get("S low"), get("V low"))
    upper = (get("H high"), get("S high"), get("V high"))
    return [(lower, upper)]


# ---------- Main loop ----------

def main():
    capture = cv2.VideoCapture(CAMERA_INDEX)
    if not capture.isOpened():
        print("Could not open the webcam. Check that no other app is using it,")
        print("or try changing CAMERA_INDEX at the top of the file.")
        return

    color_name = "blue"
    ranges = COLOR_PRESETS[color_name]
    trail = deque(maxlen=TRAIL_LENGTH)
    show_mask = True
    tuning = False

    print(__doc__)

    while True:
        ok, frame = capture.read()
        if not ok:
            print("Lost the video stream.")
            break

        frame = cv2.flip(frame, 1)   # mirror view feels more natural

        if tuning:
            ranges = read_trackbars()

        center, radius, mask = process_frame(frame, ranges)
        draw_color = DRAW_COLORS[color_name]

        if center is not None:
            cv2.circle(frame, center, radius, draw_color, 2)   # outline
            cv2.circle(frame, center, 5, (255, 255, 255), -1)  # center dot
            cv2.putText(frame, f"({center[0]}, {center[1]})",
                        (center[0] + 10, center[1] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        trail.appendleft(center)  # appendleft keeps newest first, so reverse to draw
        draw_trail(frame, list(reversed(trail)), draw_color)

        status = f"Tracking: {color_name}" + (" (custom)" if tuning else "")
        cv2.putText(frame, status, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, draw_color, 2)
        cv2.putText(frame, "1-5 color | t tune | m mask | c clear | q quit",
                    (10, frame.shape[0] - 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        cv2.imshow("Color Tracker", frame)
        if show_mask:
            cv2.imshow("Mask (what the computer sees)", mask)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key in KEY_TO_COLOR:
            color_name = KEY_TO_COLOR[key]
            ranges = COLOR_PRESETS[color_name]
            trail.clear()
            if tuning:   # reload sliders for the new color
                cv2.destroyWindow("Tuning")
                open_trackbars(ranges)
        elif key == ord("t"):
            tuning = not tuning
            if tuning:
                open_trackbars(ranges)
            else:
                cv2.destroyWindow("Tuning")
                ranges = COLOR_PRESETS[color_name]
        elif key == ord("m"):
            show_mask = not show_mask
            if not show_mask:
                cv2.destroyWindow("Mask (what the computer sees)")
        elif key == ord("c"):
            trail.clear()

    capture.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

Could not open the webcam. Check that no other app is using it,
or try changing CAMERA_INDEX at the top of the file.
